In [4]:
# -*- coding: utf-8 -*-
# 文件名: step1_feature_selection_rf_simple.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import random
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Set global random seeds
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.base import BaseEstimator, MetaEstimatorMixin, clone
from sklearn.utils.validation import check_is_fitted
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, cohen_kappa_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Configuration ---
# 🌟 核心并行设置 🌟
N_JOBS_PERF = 40  # 稳定模型 (XGB, SGBT, RF, DT, LR, SVM)
N_JOBS_STABLE = 1 # 不稳定模型 (KNN, NB, NNET)
N_JOBS_CONCURRENCY = 2 

BASE_DIR = Path(os.getcwd())

# 🌟 数据文件设置 (直接指向包含 PostopAKI 的随机森林插补文件) 🌟
IMPUTED_TRAIN_FILE = BASE_DIR / 'imputation' / 'imputed_data' / 'train_imputed_random_forest.csv'
TARGET_VARIABLE = 'PostopAKI'

# 输出设置
OUTPUT_FOLDER = BASE_DIR / 'RFE_Final_Run' 
PLOTS_FOLDER = OUTPUT_FOLDER / 'plots'
LISTS_FOLDER = OUTPUT_FOLDER / 'feature_lists'

# --- 辅助模块 ---
class PermutationImportanceWrapper(BaseEstimator, MetaEstimatorMixin):
    def __init__(self, estimator):
        self.estimator = estimator
        self.estimator_ = None 
    def fit(self, X, y, **kwargs):
        self.estimator_ = clone(self.estimator).fit(X, y, **kwargs)
        results = permutation_importance(self.estimator_, X, y, n_repeats=2, random_state=SEED, n_jobs=1)
        self.feature_importances_ = results.importances_mean
        return self
    def predict(self, X): check_is_fitted(self, "estimator_"); return self.estimator_.predict(X)
    def predict_proba(self, X): check_is_fitted(self, "estimator_"); return self.estimator_.predict_proba(X)
    @property
    def classes_(self): return self.estimator_.classes_

def get_importance_from_pipeline(fitted_pipeline):
    final_model = fitted_pipeline.named_steps['model']
    if hasattr(final_model, 'feature_importances_'): return final_model.feature_importances_
    elif hasattr(final_model, 'coef_'): return np.abs(final_model.coef_[0])
    else: raise AttributeError(f"Model {type(final_model).__name__} lacks importance attributes.")

# --- RFE 任务执行函数 ---
def run_rfe_group(model_list, X_train, y_train, n_jobs_rfe, output_folder, list_folder):
    cv_rfe = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=SEED)
    
    for name, model in model_list.items():
        print(f"  >>> [{name}] 正在运行 RFE (核心数: {n_jobs_rfe})...")
        rfe_pipeline = Pipeline([
            ('scaler', StandardScaler()), 
            ('smote', SMOTE(random_state=SEED)), 
            ('model', model)
        ])
        
        scoring_metrics = {'Accuracy': 'accuracy', 'Kappa': make_scorer(cohen_kappa_score)}
        selected_features_by_metric = {}
        
        for metric_name, scorer in scoring_metrics.items():
            try:
                rfecv = RFECV(
                    estimator=rfe_pipeline, 
                    step=1, 
                    cv=cv_rfe, 
                    scoring=scorer, 
                    min_features_to_select=5, 
                    n_jobs=n_jobs_rfe,
                    importance_getter=get_importance_from_pipeline
                )
                rfecv.fit(X_train, y_train)
                selected_features_by_metric[metric_name] = set(X_train.columns[rfecv.support_])
            except Exception as e:
                print(f"  !!! [{name}] ({metric_name}) 失败: {e}")
                selected_features_by_metric[metric_name] = set()
            
        features_acc = selected_features_by_metric.get('Accuracy', set())
        features_kappa = selected_features_by_metric.get('Kappa', set())
        combined_features = list(features_acc.intersection(features_kappa))
        
        if not combined_features:
            combined_features = list(features_acc.union(features_kappa))
        
        save_path = list_folder / f'selected_features_{name}_Combined.txt'
        with open(save_path, 'w') as f:
            for feature in sorted(combined_features): f.write(f"{feature}\n")
        
        print(f"  <<< [{name}] 完成. 筛选出 {len(combined_features)} 个特征.")

# --- 主程序 ---
def main():
    # 1. 创建文件夹
    for folder in [OUTPUT_FOLDER, PLOTS_FOLDER, LISTS_FOLDER]:
        folder.mkdir(parents=True, exist_ok=True)
            
    print("--- Data Loading (Simplified) ---")
    
    # 🌟 1. 加载插补文件 🌟
    print(f"Loading data: {IMPUTED_TRAIN_FILE}")
    if not IMPUTED_TRAIN_FILE.exists():
        print(f"❌ Error: 文件不存在 {IMPUTED_TRAIN_FILE}")
        return

    try: df_train = pd.read_csv(IMPUTED_TRAIN_FILE, encoding='gbk')
    except: df_train = pd.read_csv(IMPUTED_TRAIN_FILE, encoding='utf-8')

    # 🌟 2. 处理第一列 (无名ID列) 🌟
    # 虽然我们不需要ID来Merge，但为了整洁，我们把无名列重命名为ID，或者直接丢弃
    if df_train.columns[0].startswith('Unnamed'):
        print(f"  > Detected unnamed column 0. Renaming to 'ID'.")
        df_train.rename(columns={df_train.columns[0]: 'ID'}, inplace=True)
    
    # 🌟 3. 提取目标变量 y 🌟
    if TARGET_VARIABLE not in df_train.columns:
        raise ValueError(f"CRITICAL: '{TARGET_VARIABLE}' missing in imputed file. Please check upstream imputation.")
    
    y_train = df_train[TARGET_VARIABLE].astype(int)
    print(f"  > Target '{TARGET_VARIABLE}' extracted. Positive Rate: {y_train.mean():.4f}")

    # 🌟 4. 特征准备 (使用白名单，自动排除 ID 和 Target) 🌟
    # 即使 df_train 里有 ID，因为 ID 不在下面的列表中，所以 X 不会包含 ID
    preoperative_vars = [ 'Gender', 'Age', 'Height', 'Weight', 'BMI', 'PreopHospitalDays', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'PreopWBC', 'PreopHb', 'PreopAlb', 'PreopALT', 'PreopBUN', 'PreopCr', 'PreopGlucose' ]
    intraoperative_vars = [ 'SurgicalApproach', 'OperationTime', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopBloodLoss', 'IntraopTransfusion', 'IntraopFluid', 'IntraopHES', 'IntraopDextran', 'IntraopGelatin', 'IntraopPlasma', 'IntraopColloid', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage', 'LymphNodesExamined', 'PositiveLymphNodes' ]
    all_predictors_base = preoperative_vars + intraoperative_vars
    categorical_vars_base = ['Gender', 'WeightLoss', 'PreviousAbdominalSurgery', 'OtherMalignancy', 'Comorbidities', 'Diabetes', 'Hypertension', 'CerebrovascularDisease', 'HeartDisease', 'Smoking', 'Alcohol', 'NeoadjuvantChemo', 'ASAGrade', 'GastricColorectal', 'Gastrocolorectal', 'SurgicalApproach', 'GastricResectionSite', 'CombinedOrganResection', 'DistalGastrectomy', 'ProximalGastrectomy', 'TotalGastrectomy', 'ColorectalResectionSite', 'Stoma', 'IntraopTransfusion', 'PCAUse', 'EpiduralAnalgesia', 'IntraopDiuretics', 'IntraopVasoactive', 'CD3', 'T_Stage', 'N_Stage', 'M_Stage', 'TNM_Stage']
    
    current_predictors = [v for v in all_predictors_base if v in df_train.columns]
    current_categoricals = [v for v in categorical_vars_base if v in df_train.columns]
    current_continuous = [v for v in current_predictors if v not in current_categoricals]
    
    # 清洗连续变量 (防止读取为字符串)
    for col in current_continuous:
        df_train[col] = df_train[col].astype(str).str.extract(r'^(\d+[,.]?\d*)', expand=False)
        df_train[col] = df_train[col].str.replace(',', '.', regex=False)
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
    
    # 计算衍生指标
    required_cols = ['PLT', 'LYM', 'MONO', 'NE', 'EOS', 'BASO']
    if all(col in df_train.columns for col in required_cols):
        print("  Calculating derived features (NLR, PLR, etc.)...")
        for col in ['LYM']: df_train[col] = df_train[col].replace(0, 1e-6)
        new_features = ['PLR', 'MLR', 'ELR', 'BLR', 'NLR', 'SII']
        df_train['PLR'], df_train['MLR'], df_train['ELR'], df_train['BLR'], df_train['NLR'], df_train['SII'] = \
            df_train['PLT']/df_train['LYM'], df_train['MONO']/df_train['LYM'], df_train['EOS']/df_train['LYM'], \
            df_train['BASO']/df_train['LYM'], df_train['NE']/df_train['LYM'], df_train['PLT']*df_train['NE']/df_train['LYM']
        current_predictors.extend([f for f in new_features if f not in current_predictors])
    
    # 🌟 5. 构建特征矩阵 X 🌟
    # 只包含 current_predictors 列表中的列，绝对安全！
    X_train_full_raw = df_train[current_predictors]
    X_train_full_dummies = pd.get_dummies(X_train_full_raw, columns=current_categoricals, drop_first=True)
    X_train = X_train_full_dummies.copy()
    
    if X_train.isnull().sum().sum() > 0:
        print(f"  Warning: Filling {X_train.isnull().sum().sum()} remaining NaNs with median.")
        X_train.fillna(X_train.median(), inplace=True)

    print(f"Ready for RFE.")
    print(f"  - Features: {len(X_train.columns)}")
    print(f"  - Samples: {len(X_train)}")
    
    # 再次确认数据对齐 (同一文件读取，必然对齐)
    if len(X_train) != len(y_train):
        raise ValueError("Critical Error: Length mismatch between X and y!")

    # 6. 定义模型组
    models_full = {
        'LR': LogisticRegression(max_iter=1000, random_state=SEED), 
        'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED),
        'KNN': PermutationImportanceWrapper(KNeighborsClassifier()),
        'SVM': LinearSVC(random_state=SEED, dual=False, class_weight='balanced', max_iter=2000), 
        'NB': PermutationImportanceWrapper(GaussianNB()),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': PermutationImportanceWrapper(MLPClassifier(max_iter=500, random_state=SEED))
    }
    
    # 7. 划分并发组
    unstable_models = ['KNN', 'NB', 'NNET']
    models_stable = {k: v for k, v in models_full.items() if k in unstable_models}
    models_perf = {k: v for k, v in models_full.items() if k not in unstable_models}
    
    # 8. 并发执行
    print("\n--- 开始 RFE 特征选择 (并发模式) ---")
    
    with ThreadPoolExecutor(max_workers=N_JOBS_CONCURRENCY) as executor:
        future_perf = executor.submit(run_rfe_group, models_perf, X_train, y_train, N_JOBS_PERF, OUTPUT_FOLDER, LISTS_FOLDER)
        future_stable = executor.submit(run_rfe_group, models_stable, X_train, y_train, N_JOBS_STABLE, OUTPUT_FOLDER, LISTS_FOLDER)
        future_perf.result()
        future_stable.result()
        
    print("\n--- ✅ 所有模型的特征选择已完成! ---")
    print(f"结果已保存至: {LISTS_FOLDER}")

if __name__ == '__main__':
    main()

--- Data Loading (Simplified) ---
Loading data: /mnt/d/AKI新分析/imputation/imputed_data/train_imputed_random_forest.csv
  > Detected unnamed column 0. Renaming to 'ID'.
  > Target 'PostopAKI' extracted. Positive Rate: 0.0298
Ready for RFE.
  - Features: 15
  - Samples: 2446

--- 开始 RFE 特征选择 (并发模式) ---
  >>> [LR] 正在运行 RFE (核心数: 40)...
  >>> [KNN] 正在运行 RFE (核心数: 1)...
  <<< [LR] 完成. 筛选出 10 个特征.
  >>> [DT] 正在运行 RFE (核心数: 40)...
  <<< [DT] 完成. 筛选出 6 个特征.
  >>> [RF] 正在运行 RFE (核心数: 40)...
  <<< [RF] 完成. 筛选出 14 个特征.
  >>> [SVM] 正在运行 RFE (核心数: 40)...
  <<< [SVM] 完成. 筛选出 10 个特征.
  >>> [XGB] 正在运行 RFE (核心数: 40)...


[21:19:11] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:19:26] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:19:42] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:19:58] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:20:14] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:20:29] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:20:45] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:21:00] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:21:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[21:21:33] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_la

  <<< [XGB] 完成. 筛选出 8 个特征.
  >>> [SGBT] 正在运行 RFE (核心数: 40)...
  <<< [SGBT] 完成. 筛选出 7 个特征.
  <<< [KNN] 完成. 筛选出 13 个特征.
  >>> [NB] 正在运行 RFE (核心数: 1)...
  <<< [NB] 完成. 筛选出 5 个特征.
  >>> [NNET] 正在运行 RFE (核心数: 1)...
  <<< [NNET] 完成. 筛选出 11 个特征.

--- ✅ 所有模型的特征选择已完成! ---
结果已保存至: /mnt/d/AKI新分析/RFE_Final_Run/feature_lists
